# Fine-tuning GLiNER pentru NER Politico-Economic

Acest notebook antreneaza un model GLiNER pe datasetul construit anterior si:

1. Incarca dataset-ul `train.jsonl` / `dev.jsonl` / `test.jsonl`
2. Converteste din format char-span in format GLiNER (`tokenized_text` + `ner` cu token indices)
3. Fine-tuning peste un model GLiNER pre-antrenat (`urchade/gliner_small-v2.1`)
4. Evalueaza pe test set: precision, recall, F1 (per-label si micro/macro)

Macro = se calculeaza metricile pe fiecare label si apoi se face media lor

Micro = se aduna toate TP, FP, FN ale claselor si apoi se calculeaza metricile o singura data. 

5. Incarca dataset-ul pe HuggingFace (repo: `Tudorx95/NER_Political_Economic`)
6. Incarca modelul antrenat pe HuggingFace (repo: `Tudorx95/NER_Economic_Political`)

**Cerinte hardware:** GPU recomandat (T4 in Colab e suficient pt `gliner_small-v2.1`).


## 1. Instalare dependinte

In [ ]:
# Instalare dependinte (Colab)
!pip install -q gliner==0.2.13 accelerate transformers datasets seqeval nervaluate huggingface_hub
!pip install --upgrade gliner huggingface_hub transformers
print("Instalare completa. Daca apar warning-uri, restarteaza runtime: Runtime > Restart Runtime.")

  Using cached gliner-0.2.26-py3-none-any.whl.metadata (10 kB)
  Using cached transformers-5.6.2-py3-none-any.whl.metadata (33 kB)
Using cached gliner-0.2.26-py3-none-any.whl (170 kB)
  Attempting uninstall: gliner
    Found existing installation: gliner 0.2.13
    Uninstalling gliner-0.2.13:
      Successfully uninstalled gliner-0.2.13
Instalare completa. Daca apar warning-uri, restarteaza runtime: Runtime > Restart Runtime.


## 2. Configurare globala

Ajusteaza `BASE_DIR` astfel incat sa coincida cu locul unde ai salvat dataset-ul in notebook-ul anterior.

Tokenizer = componenta care imparte raw text de input in tokens pe care modelul sa ii inteleaga
            ("converts raw text to sparse index encodings") 

          - asociaza cuvintele (tokens) cu labelurile definite de model.

### De ce modelul gliner_small-v2.1 si nu gliner_medium-v2.5 ?

Modelul gliner_small-v2.1 foloseste drept encoder DeBERTa-v3-small (~70M parametri, 6 layers, hidden size 384).

Modelul gliner_medium-v2.5 foloseste drept encoder DeBERTa-v3-base (~183M parametri, 12 layers, hidden size 768). 

Scopul alegerii modelelor Gliner este legat de zero shot learning. Daca avem un model cu prea mult context, s-ar putea ca etichetele noi sau cele cu putine exemple in setul de date sa nu mai fie prezise corect fara o antrenare atat de buna. 

Un model cu mai mult context nu va putea compensia lipsa de date pentru antrenarea unui model specific. 

In [ ]:
import os
from pathlib import Path

# Mount the Drive 
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = Path("/content/drive/MyDrive/NER_Project")

SPLITS_DIR    = BASE_DIR / "splits"                 # dir for train/val/test splits
MODEL_OUT_DIR = BASE_DIR / "gliner_finetuned"       # dir for saving the fine-tuned model and tokenizer

MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)    

PRETRAINED_MODEL = "urchade/gliner_small-v2.1"  # alternative: gliner-community/gliner_medium-v2.5
NUM_EPOCHS       = 10
BATCH_SIZE       = 8
LEARNING_RATE    = 3e-6

WARMUP_RATIO     = 0.15     # procentul initial din pasii de antrenare pt care lr creste treptat pana la valoarea specificata
                            # previne gradienti instabili la inceputul antrenamentului
WEIGHT_DECAY     = 0.05     # termen de regularizare ce penalizeaza ponderile mari (previne overfitting)
MAX_GRAD_NORM    = 1.0      # pt gradient clipping (if norm gradient > MAX_GRAD_NORM, se scaleaza gradientii astfel incat norm = MAX_GRAD_NORM)
EVAL_THRESHOLD   = 0.5      # prag folosit la inferenta pt a decide daca predictia e acceptata sau nu (if confidence > EVAL_THRESHOLD, accepta predictia else o respinge)


# HuggingFace repos 
HF_DATASET_REPO = "Tudorx95/NER_Political_Economic"
HF_MODEL_REPO   = "Tudorx95/NER_Economic_Political"

# Schema NER
NER_LABELS = [
    "POLITICIAN", "POLITICAL_PARTY", "POLITICAL_ORG", "FINANCIAL_ORG",
    "ECONOMIC_INDICATOR", "POLICY", "LEGISLATION", "MARKET_EVENT",
    "CURRENCY", "TRADE_AGREEMENT", "GPE",  # Geopolitical Entity
]

print(f"BASE_DIR        = {BASE_DIR}")
print(f"MODEL_OUT_DIR   = {MODEL_OUT_DIR}")
print(f"PRETRAINED      = {PRETRAINED_MODEL}")
print(f"HF dataset repo = {HF_DATASET_REPO}")
print(f"HF model repo   = {HF_MODEL_REPO}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE_DIR        = /content/drive/MyDrive/NER_Project
MODEL_OUT_DIR   = /content/drive/MyDrive/NER_Project/gliner_finetuned
PRETRAINED      = urchade/gliner_small-v2.1
HF dataset repo = Tudorx95/NER_Political_Economic
HF model repo   = Tudorx95/NER_Economic_Political


## 3. Verificare GPU

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device:          {torch.cuda.get_device_name(0)}")
    print(f"Memory total:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = torch.device("cuda:0")
else:
    print("ATENTIE: rulezi pe CPU. Antrenarea va fi LENTA. Activeaza GPU in Colab: Runtime > Change runtime type > GPU.")
    DEVICE = torch.device("cpu")

PyTorch version: 2.10.0+cu128
CUDA available:  True
Device:          Tesla T4
Memory total:    15.6 GB


## 4. Incarcare dataset si conversie in format GLiNER

GLiNER asteapta urmatorul format pentru fiecare exemplu:

```python
{
    "tokenized_text": ["The", "Federal", "Reserve", "raised", "rates", "."],
    "ner": [[1, 2, "FINANCIAL_ORG"]]  # [start_token_idx, end_token_idx (inclusiv), label]
}
```

Dataset-ul nostru este in format char-span:
```python
{
    "text": "The Federal Reserve raised rates.",
    "entities": [{"text": "Federal Reserve", "label": "FINANCIAL_ORG", "start": 4, "end": 19}]
}
```

Trebuie sa convertim. Folosim un tokenizer simplu (whitespace + punctuatie) si mapam char positions in token indices.

In [ ]:
import json
import re
from typing import List, Dict, Tuple

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

# Tokenizer simplu compatibil cu GLiNER
TOKEN_PATTERN = re.compile(
    r"(?:[A-Za-z]\.){2,}"          # abrevieri: U.S., U.K., e.g., St.
    r"|\w+(?:[-']\w+)*"            # cuvinte cu cratima/apostrof: Kwazulu-Natal, don't
    r"|[^\w\s]"                    # punctuatie ramasa
)

# Returneaza lista de tokeni impreuna cu offset-urile lor in text: [(token, char_start, char_end), ...]
# Prin acest Tokenizer se extrag obiectele din lista "entities"

def tokenize_with_offsets(text: str) -> List[Tuple[str, int, int]]:
    """Returneaza [(token, char_start, char_end), ...]."""
    return [(m.group(), m.start(), m.end()) for m in TOKEN_PATTERN.finditer(text)]
    # aici efectiv group = token/cuvant, start = pozitia de inceput din text, end = pozitia de sfarsit din text (exclusive)

def convert_to_gliner_format(example: Dict) -> Dict:
    """Converteste un exemplu char-span in format GLiNER token-span."""
    # Pt fiecare obiect din fisierul de intrare, se extrage textul si obiectele din "entities". 
    text = example["text"]                                                              # text-ul original
    tokens_with_offsets = tokenize_with_offsets(text)                                   # sparge textul in tokeni si retine offset-urile lor in text
    tokens = [t for t, _, _ in tokens_with_offsets]                                     # extrage doar tokenii (fara offset-uri)

    ner = []
    # se cauta pentru fiecare entitate care sunt tokenii care corespund span-ului de caractere al entitatii.
    for ent in example.get("entities", []):
        # extragem pozitiile de start si end din entitate. 
        ent_start_char = ent["start"]
        ent_end_char   = ent["end"]
        # Cautam token care incepe la ent_start_char (sau cel mai apropiat dupa)
        start_tok = None
        end_tok = None
        # se verifica in textul original daca exista tokeni care incep sau se termina exact la pozitiile de start si end ale entitatii. Daca da, se retin indexii acelor tokeni.
        for i, (_, cs, ce) in enumerate(tokens_with_offsets):
            if start_tok is None and cs >= ent_start_char:
                start_tok = i       # indexul cuvantului in text 
            if ce <= ent_end_char:
                end_tok = i

        # Fallback: in caz ca nu exista tokenul in text, atunci cautam tokeni care se suprapun cu span-ul de caractere al entitatii (cs < ent_end_char si ce > ent_start_char). 
        # Daca gasim astfel de tokeni, retinem indexul primului gasit ca start_tok si indexul ultimului gasit ca end_tok.
        if start_tok is None or end_tok is None or start_tok > end_tok:
            for i, (_, cs, ce) in enumerate(tokens_with_offsets):
                if cs < ent_end_char and ce > ent_start_char:
                    if start_tok is None:
                        start_tok = i
                    end_tok = i
        # daca am gasit un token care apare in text, il adaugam in lista (pozitia cuvantului in text!!!).
        if start_tok is not None and end_tok is not None and start_tok <= end_tok:
            ner.append([start_tok, end_tok, ent["label"]])

    return {"tokenized_text": tokens, "ner": ner}

# Se returneaza un dict cu "tokenized_text" (lista de tokeni) si "ner" (lista de entitati in format [start_token_idx, end_token_idx, label]).

# Incarcam dataset-urile
train_raw = load_jsonl(SPLITS_DIR / "train.jsonl")
dev_raw   = load_jsonl(SPLITS_DIR / "dev.jsonl")
test_raw  = load_jsonl(SPLITS_DIR / "test.jsonl")

print(f"Raw counts: train={len(train_raw)}, dev={len(dev_raw)}, test={len(test_raw)}")

# Conversie
train_data = [convert_to_gliner_format(ex) for ex in train_raw]
dev_data   = [convert_to_gliner_format(ex) for ex in dev_raw]
test_data  = [convert_to_gliner_format(ex) for ex in test_raw]

# Filtram exemplele care au pierdut toate entitatile la conversie
train_data = [ex for ex in train_data if ex["ner"]]
dev_data   = [ex for ex in dev_data   if ex["ner"]]
test_data  = [ex for ex in test_data  if ex["ner"]]

print(f"Dupa conversie + filtrare:")
print(f"  train: {len(train_data)}")
print(f"  dev:   {len(dev_data)}")
print(f"  test:  {len(test_data)}")

# Sanity check pe primul exemplu
print("\nExemplu convertit:")
print(json.dumps(train_data[0], indent=2)[:400])

Raw counts: train=5747, dev=1228, test=2124
Dupa conversie + filtrare:
  train: 5747
  dev:   1228
  test:  2124

Exemplu convertit:
{
  "tokenized_text": [
    "In",
    "early",
    "April",
    ",",
    "US",
    "President",
    "Donald",
    "Trump",
    "ordered",
    "the",
    "assault",
    "group",
    "of",
    "USS",
    "Carl",
    "Vinson",
    "aircraft",
    "carrier",
    "to",
    "go",
    "to",
    "North",
    "Korea",
    "and",
    "stop",
    "near",
    "the",
    "Korean",
    "Peninsula",
    "in",
  


### Metoda 2 de conversie folosind AutoTokenizer

In [ ]:
# sau puteam folosi direct un Tokenizer 
from typing import Dict
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def convert_to_gliner_format(example: Dict) -> Dict:
    text = example["text"]

    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        add_special_tokens=False
    )

    tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
    offsets = encoding["offset_mapping"]

    ner = []
    for ent in example.get("entities", []):
        start_char = ent["start"]
        end_char = ent["end"]

        start_tok = end_tok = None

        for i, (s, e) in enumerate(offsets):
            if start_tok is None and s <= start_char < e:
                start_tok = i
            if s < end_char <= e:
                end_tok = i

        if start_tok is None or end_tok is None or start_tok > end_tok:
            for i, (s, e) in enumerate(offsets):
                if s < end_char and e > start_char:
                    if start_tok is None:
                        start_tok = i
                    end_tok = i

        if start_tok is not None and end_tok is not None:
            ner.append([start_tok, end_tok, ent["label"]])
    
    return {"tokenized_text": tokens, "ner": ner}

## 5. Verificare conversie

Verificam ca span-urile token-based reconstituie corect entitatile originale. O mica vedere asupra lor. 

In [ ]:
print(train_raw[0])
print(train_data[0])

{'text': 'In early April, US President Donald Trump ordered the assault group of USS Carl Vinson aircraft carrier to go to North Korea and stop near the Korean Peninsula in the western part of the Pacific Ocean.', 'entities': [{'text': 'Donald Trump', 'label': 'POLITICIAN', 'start': 29, 'end': 41}], 'source': 'weak_supervision_cc_news', 'snorkel_confidence': 0.9104803965467734}
{'tokenized_text': ['In', 'early', 'April', ',', 'US', 'President', 'Donald', 'Trump', 'ordered', 'the', 'assault', 'group', 'of', 'USS', 'Carl', 'Vinson', 'aircraft', 'carrier', 'to', 'go', 'to', 'North', 'Korea', 'and', 'stop', 'near', 'the', 'Korean', 'Peninsula', 'in', 'the', 'western', 'part', 'of', 'the', 'Pacific', 'Ocean', '.'], 'ner': [[6, 7, 'POLITICIAN']]}


In [ ]:
real_discrepancies = 0

for i in range(len(train_data)):
    train_r = train_raw[i]['entities']
    train_d = train_data[i]['ner']
    tokens  = train_data[i]['tokenized_text']

    # Reconstruim textul fiecarei entitati din GLiNER format
    gliner_texts = {" ".join(tokens[s:e+1]) for s, e, lbl in train_d}
    raw_texts    = {ent['text'].strip() for ent in train_r}

    # Comparam textele, nu pozitiile
    lost_words = raw_texts - gliner_texts      # in raw dar nu in gliner
    added_words = gliner_texts - raw_texts      # in gliner dar nu in raw

    if lost_words or added_words:
        real_discrepancies += 1
        if real_discrepancies <= 5:  # afisam doar primele 5
            print(f"\n🔴 Exemplul {i}")
            print(f"  Pierdute la conversie: {lost_words}")
            print(f"  Adaugate eronat:       {added_words}")
            print(f"  Text: {train_raw[i]['text'][:100]}")

print(f"\nTotal discrepante reale: {real_discrepancies} / {len(train_data)}")

Verificare conversie pe 5 exemple:

--- Exemplul 0 ---
Text:           In early April , US President Donald Trump ordered the assault group of USS Carl Vinson aircraft car...
Entitati gasite: [('Donald Trump', 'POLITICIAN')]
Entitati orig:   [('Donald Trump', 'POLITICIAN')]

--- Exemplul 1 ---
Text:           Gallagher made his usual obscene gestures and swore at journalists as he prepared to fly from London...
Entitati gasite: [('London', 'GPE'), ('Heathrow', 'GPE'), ('Chicago', 'GPE')]
Entitati orig:   [('London', 'GPE'), ('Heathrow', 'GPE'), ('Chicago', 'GPE')]

--- Exemplul 2 ---
Text:           Winger Lee Sharpe hit a superb strike from the edge of the penalty area to give Leeds their first wi...
Entitati gasite: [('England', 'GPE')]
Entitati orig:   [('England', 'GPE')]

--- Exemplul 3 ---
Text:           Niugini holds copper and gold mining interests in Australia , Chile and Papua New Guinea , where it ...
Entitati gasite: [('Australia', 'GPE'), ('Chile', 'GPE'), ('Papua New Gui

## 6. Incarcare model pre-antrenat GLiNER

In [ ]:
from gliner import GLiNER

print(f"Se incarca {PRETRAINED_MODEL}...")
model = GLiNER.from_pretrained(PRETRAINED_MODEL)
model.to(DEVICE)

# Test inferenta zero-shot inainte de fine-tuning
test_text = "The Federal Reserve raised interest rates by 25 basis points after the FOMC meeting, citing inflation concerns. President Biden welcomed the decision."
zero_shot_entities = model.predict_entities(test_text, NER_LABELS, threshold=EVAL_THRESHOLD)

print(f"\nPredictii zero-shot (inainte de fine-tuning):")
for ent in zero_shot_entities:
    print(f"  {ent['text']:<35} -> {ent['label']:<20} (score: {ent['score']:.3f})")

Se incarca urchade/gliner_small-v2.1...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]


Predictii zero-shot (inainte de fine-tuning):
  Federal Reserve                     -> FINANCIAL_ORG        (score: 0.963)
  FOMC meeting                        -> MARKET_EVENT         (score: 0.643)
  President Biden                     -> POLITICIAN           (score: 0.887)


Cum arata Gliner config.json:

```
{
  "size_sup": -1,
  "max_types": 25,
  "shuffle_types": true,
  "random_drop": true,
  "max_neg_type_ratio": 1,
  "max_len": 384,
  "lr_encoder": "1e-5",
  "lr_others": "5e-5",
  "num_steps": 30000,
  "warmup_ratio": 3000,
  "train_batch_size": 8,
  "eval_every": 5000,
  "max_width": 12,
  "model_name": "microsoft/deberta-v3-small",
  "fine_tune": true,
  "subtoken_pooling": "first",
  "hidden_size": 512,
  "span_mode": "markerV0",
  "dropout": 0.4,
  "name": "correct"
}
```

## 7. Setup training

Folosim API-ul oficial GLiNER cu `Trainer` si `TrainingArguments`.

In [ ]:
from gliner.training import Trainer, TrainingArguments
from gliner.data_processing.collator import SpanDataCollator

# ─── Data collator — versiunea corecta pentru GLiNER >= 0.2.14 ───────────────
# DataCollator a fost eliminat. Modelele span-based (gliner_small, gliner_medium)
# folosesc SpanDataCollator. Modelele token-based folosesc TokenDataCollator.
data_collator = SpanDataCollator(
    config=model.config,                    # contine configuratia modelului (backbone: Bert, RoBerta, etc. ; hidden size, nb layers, etc.)
    data_processor=model.data_processor,    # contine logica de preprocesare a datelor (converteste text+entitati in formatul necesar modelului)
    prepare_labels=True,
)

# Calcul total steps pentru scheduler
num_steps = (len(train_data) // BATCH_SIZE) * NUM_EPOCHS           # numarul total de update-uri pe care le va face modelul in timpul trainingului
num_warmup = int(num_steps * WARMUP_RATIO)                         # numarul de update-uri pentru care lr creste treptat la inceputul antrenamentului (previne gradienti instabili)
print(f"Total training steps: {num_steps}, warmup: {num_warmup}")

training_args = TrainingArguments(
    output_dir=str(MODEL_OUT_DIR / "checkpoints"),  # unde se salveaza checkpoint-urile in timpul trainingului
    learning_rate=LEARNING_RATE,                
    weight_decay=WEIGHT_DECAY,                      # termen de regularizare ce penalizeaza ponderile mari (previne overfitting)
    others_lr=1e-5,                                 # learning rate mic pentru restul parametrilor (ex: backbone) pentru a nu strica cunostintele deja invatate in pretraining
    others_weight_decay=0.01,                       # termen de regularizare pentru restul parametrilor
    lr_scheduler_type="linear",                     # scheduler liniar care scade lr de la valoarea initiala la 0 pe parcursul trainingului
    warmup_steps=WARMUP_RATIO,                      # procentul initial din pasii de antrenare pt care lr creste treptat pana la valoarea specificata
    per_device_train_batch_size=BATCH_SIZE,         # numarul de exemple procesate simultan pe fiecare dispozitiv (GPU/CPU) in timpul trainingului
    per_device_eval_batch_size=BATCH_SIZE,          # numarul de exemple procesate simultan pe fiecare dispozitiv (GPU/CPU) in timpul evaluarii
    num_train_epochs=NUM_EPOCHS,
    eval_strategy="epoch",                          # evaluation_strategy in versiunile vechi
    save_strategy="epoch",
    dataloader_num_workers=0,                       # numarul de subprocesses folosite pentru a incarca datele (0 = se incarca in procesul principal, >0 = se folosesc subprocesses pentru a incarca datele in paralel)
    use_cpu=(DEVICE.type == "cpu"),
    report_to="none",
    logging_steps=50,
    max_grad_norm=MAX_GRAD_NORM,                    # pt gradient clipping (if norm gradient > MAX_GRAD_NORM, se scaleaza gradientii astfel incat norm = MAX_GRAD_NORM)
    remove_unused_columns=False,                    # CRITIC: fara asta Trainer sterge coloanele custom (ex: "tokenized_text", "ner") din dataset, ceea ce strica logica de preprocesare a datelor din data_processor. Seteaza False pentru a pastra toate coloanele din dataset.
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,                        # pentru ca vrem sa minimizam pierderea (loss) la evaluare
    save_total_limit=3,        # pastreaza top 3 checkpoints
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=dev_data,
    processing_class=model.data_processor.transformer_tokenizer,  # 'tokenizer' -> 'processing_class'
    data_collator=data_collator,
)

print("Trainer configurat cu succes!")
print(f"  Collator:  {type(data_collator).__name__}")
print(f"  Processor: {type(model.data_processor).__name__}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Total training steps: 7180, warmup: 1436
Trainer configurat cu succes!
  Collator:  SpanDataCollator
  Processor: UniEncoderSpanProcessor


## 8. Antrenare

In [ ]:
print("Incepe antrenarea...\n")
trainer.train()
print("\nAntrenare completa.")

# Salvam modelul final
final_model_path = MODEL_OUT_DIR / "final"
model.save_pretrained(str(final_model_path))
print(f"\nModel salvat in: {final_model_path}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1, 'pad_token_id': 0}.


Incepe antrenarea...



Epoch,Training Loss,Validation Loss
1,14.946473,14.350679
2,10.853578,10.571704
3,9.216663,9.525786
4,7.977938,9.072790
5,7.296552,8.452920
6,7.055439,8.590841
7,7.104838,7.699710
8,5.924961,8.275341
9,5.652329,8.186038
10,6.648803,8.344475


/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:392: UserWarning: Sentence of length 22993 has been truncated to 384
  self.preprocess_example(b["tokenized_text"], b[key], class_to_ids[i]) for i, b in enumerate(batch_list)
/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:392: UserWarning: Sentence of length 22993 has been truncated to 384
  self.preprocess_example(b["tokenized_text"], b[key], class_to_ids[i]) for i, b in enumerate(batch_list)
/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:392: UserWarning: Sentence of length 22993 has been truncated to 384
  self.preprocess_example(b["tokenized_text"], b[key], class_to_ids[i]) for i, b in enumerate(batch_list)
/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:392: UserWarning: Sentence of length 22993 has been truncated to 384
  self.preprocess_example(b["tokenized_text"], b[key], class_to_ids[i]) for i, b in enumerate(batch_li


Antrenare completa.

Model salvat in: /content/drive/MyDrive/NER_Project/gliner_finetuned/final


## 9. Evaluare pe test set

Calculam metrici NER standard: precision, recall, F1 (per-label si micro/macro).
Folosim `nervaluate` pentru evaluare span-based.

In [ ]:
# Reincarcam modelul salvat pentru evaluare curata
from gliner import GLiNER

eval_model = GLiNER.from_pretrained(str(final_model_path), local_files_only=True) # incarcam modelul din calea locala (fara a incerca sa-l descarce de pe HuggingFace Hub)
eval_model.to(DEVICE)
eval_model.eval()           # seteaza modelul in modul de evaluare (dezactiveaza dropout, etc.) pentru a obtine predictii consistente la inferenta

# Convertim test set inapoi in format text + entitati char-span pentru evaluare
def gliner_to_char_spans(ex):
    """Reconstruieste textul cu spatii intre tokens si calculeaza char spans."""
    tokens = ex["tokenized_text"]   
    text_parts = []
    char_pos = 0
    token_starts, token_ends = [], []
    for i, t in enumerate(tokens):
        if i > 0:                   # daca nu e primul cuvant
            text_parts.append(" ")
            char_pos += 1
        # recalculeaza pozitiile si adauga in liste 
        token_starts.append(char_pos)   
        text_parts.append(t)
        char_pos += len(t)
        token_ends.append(char_pos)
    # reconstruieste textul
    text = "".join(text_parts)
    entities = []
    for s_tok, e_tok, lbl in ex["ner"]:
        entities.append({
            "text": text[token_starts[s_tok]:token_ends[e_tok]],
            "label": lbl,
            "start": token_starts[s_tok],
            "end": token_ends[e_tok],
        })
    return text, entities

# Generam predictii pe test set
print(f"Se evalueaza pe {len(test_data)} exemple test...")
gold_entities_per_doc = []
pred_entities_per_doc = []

from tqdm.auto import tqdm
for ex in tqdm(test_data, desc="Predictie test"):   # parcurgem fiecare exemplu din test_data
    text, gold = gliner_to_char_spans(ex)           # reconstruieste textul original si entitatile gold in format char-span
    pred_raw = eval_model.predict_entities(text, NER_LABELS, threshold=EVAL_THRESHOLD)      # obtine predictiile modelului in format char-span (lista de dicturi cu "text", "label", "start", "end")
    # Format pred pentru nervaluate
    pred = [{"start": p["start"], "end": p["end"], "label": p["label"]} for p in pred_raw]
    gold_n = [{"start": g["start"], "end": g["end"], "label": g["label"]} for g in gold]
    gold_entities_per_doc.append(gold_n)
    pred_entities_per_doc.append(pred)

print(f"\nDocumente evaluate: {len(gold_entities_per_doc)}")
print(f"Total entitati gold: {sum(len(g) for g in gold_entities_per_doc)}")
print(f"Total entitati pred: {sum(len(p) for p in pred_entities_per_doc)}")

Se evalueaza pe 2124 exemple test...


Predictie test:   0%|          | 0/2124 [00:00<?, ?it/s]


Documente evaluate: 2124
Total entitati gold: 3224
Total entitati pred: 4168


In [ ]:
print(f"Se evalueaza pe {len(test_raw)} exemple test...")
gold_entities_per_doc = []
pred_entities_per_doc = []

for ex in tqdm(test_raw, desc="Predictie test"):
    # Folosim textul ORIGINAL, nu reconstruit din tokeni
    text = ex["text"]
    
    # Gold direct din test_raw — char spans originale, fara conversie
    gold_n = [
        {"start": e["start"], "end": e["end"], "label": e["label"]}
        for e in ex.get("entities", [])
    ]
    
    # Predictii pe textul original
    pred_raw = eval_model.predict_entities(text, NER_LABELS, threshold=EVAL_THRESHOLD)
    pred = [
        {"start": p["start"], "end": p["end"], "label": p["label"]}
        for p in pred_raw
    ]
    
    gold_entities_per_doc.append(gold_n)
    pred_entities_per_doc.append(pred)

print(f"\nDocumente evaluate: {len(gold_entities_per_doc)}")
print(f"Total entitati gold: {sum(len(g) for g in gold_entities_per_doc)}")
print(f"Total entitati pred: {sum(len(p) for p in pred_entities_per_doc)}")

In [ ]:
# ─── Evaluare cu nervaluate 1.2.x ────────────────────────────────────────────
# EvaluationResult foloseste atribute directe (.precision, .recall, .f1)
# NU subscript ["precision"] — acesta e motivul erorii TypeError
from nervaluate import Evaluator

evaluator = Evaluator(
    gold_entities_per_doc,
    pred_entities_per_doc,
    tags=NER_LABELS,
)
result = evaluator.evaluate()

results         = result["overall"]   # metrici globale
results_per_tag = result["entities"]  # metrici per label

# ─── Metrici globale ─────────────────────────────────────────────────────────
print("=" * 70)
print("METRICI GLOBALE (micro-averaged pe toate labels)")
print("=" * 70)
for mode in ["ent_type", "partial", "strict", "exact"]:
    r = results[mode]  # r este EvaluationResult — acces prin atribute, nu dict
    print(f"\nMod '{mode}':")
    print(f"  Precision: {r.precision:.4f}")
    print(f"  Recall:    {r.recall:.4f}")
    print(f"  F1:        {r.f1:.4f}")

# ─── Metrici per label ───────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("METRICI PER LABEL (mod 'ent_type' = label match)")
print("=" * 70)
print(f"{'Label':<22} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
print("-" * 70)

for label in NER_LABELS:
    support = sum(1 for doc in gold_entities_per_doc for e in doc if e["label"] == label)
    if label in results_per_tag:
        m = results_per_tag[label]["ent_type"]
        print(f"{label:<22} {m.precision:>10.4f} {m.recall:>10.4f} {m.f1:>10.4f} {support:>10}")
    else:
        print(f"{label:<22} {'N/A':>10} {'N/A':>10} {'N/A':>10} {support:>10}")

METRICI GLOBALE (micro-averaged pe toate labels)

Mod 'ent_type':
  Precision: 0.6833
  Recall:    0.8834
  F1:        0.7706

Mod 'partial':
  Precision: 0.7185
  Recall:    0.9288
  F1:        0.8102

Mod 'strict':
  Precision: 0.6730
  Recall:    0.8700
  F1:        0.7589

Mod 'exact':
  Precision: 0.7097
  Recall:    0.9175
  F1:        0.8003

METRICI PER LABEL (mod 'ent_type' = label match)
Label                   Precision     Recall         F1    Support
----------------------------------------------------------------------
POLITICIAN                 0.6118     0.8902     0.7252        501
POLITICAL_PARTY            0.7293     0.9635     0.8302        137
POLITICAL_ORG              0.3417     0.3636     0.3523        187
FINANCIAL_ORG              0.2080     0.5000     0.2938        104
ECONOMIC_INDICATOR         0.1000     0.2000     0.1333          5
POLICY                     0.0000     0.0000     0.0000          4
LEGISLATION                0.1429     1.0000     0.2500    

In [ ]:
# ─── Salvare metrici in fisier ──────────────────────────────────────────────
import json
from datetime import datetime

metrics_summary = {
    "timestamp": datetime.now().isoformat(),
    "pretrained_model": PRETRAINED_MODEL,
    "num_train_examples": len(train_data),
    "num_dev_examples":   len(dev_data),
    "num_test_examples":  len(test_data),
    "training_config": {
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "warmup_ratio": WARMUP_RATIO,
        "weight_decay": WEIGHT_DECAY,
    },
    "eval_threshold": EVAL_THRESHOLD,
    "global_metrics": {
        mode: {
            "precision": float(results[mode].precision),
            "recall":    float(results[mode].recall),
            "f1":        float(results[mode].f1),
        }
        for mode in ["ent_type", "partial", "strict", "exact"]
    },
    "per_label_metrics": {
        label: {
            mode: {
                "precision": float(results_per_tag[label][mode].precision),
                "recall":    float(results_per_tag[label][mode].recall),
                "f1":        float(results_per_tag[label][mode].f1),
            }
            for mode in ["ent_type", "strict"]
        }
        for label in NER_LABELS if label in results_per_tag
    },
}

metrics_path = MODEL_OUT_DIR / "metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics_summary, f, indent=2)
print(f"Metrici salvate: {metrics_path}")
print(f"\nF1 global (ent_type): {metrics_summary['global_metrics']['ent_type']['f1']:.4f}")
print(f"F1 global (strict):   {metrics_summary['global_metrics']['strict']['f1']:.4f}")

Metrici salvate: /content/drive/MyDrive/NER_Project/gliner_finetuned/metrics.json

F1 global (ent_type): 0.7706
F1 global (strict):   0.7589


## 10. Test calitativ — predictii pe propozitii noi

Sa vedem cum se comporta modelul fine-tunat pe texte noi.

In [ ]:
# Texte de test reprezentative pentru domeniul nostru
test_sentences = [
    "The Federal Reserve raised interest rates by 50 basis points, citing persistent inflation pressures.",
    "President Biden signed the Inflation Reduction Act into law after months of congressional negotiations.",
    "The European Central Bank announced a new round of quantitative easing to support the eurozone economy.",
    "Christine Lagarde said the ECB remains committed to its 2% inflation target despite rising energy costs.",
    "NATO members agreed to increase defense spending following the latest G20 summit in Tokyo.",
    "Goldman Sachs analysts predict the dollar will weaken against the yen in the coming quarter.",
    "The Republican Party secured a narrow majority in the Senate after the midterm elections.",
]

print("PREDICTII MODEL FINE-TUNAT")
print("=" * 70)
for sent in test_sentences:
    print(f"\nText: {sent}")
    entities = eval_model.predict_entities(sent, NER_LABELS, threshold=EVAL_THRESHOLD)
    for ent in entities:
        print(f"  {ent['text']:<35} -> {ent['label']:<20} (score: {ent['score']:.3f})")

PREDICTII MODEL FINE-TUNAT

Text: The Federal Reserve raised interest rates by 50 basis points, citing persistent inflation pressures.
  Federal Reserve                     -> FINANCIAL_ORG        (score: 0.998)

Text: President Biden signed the Inflation Reduction Act into law after months of congressional negotiations.
  Biden                               -> POLITICIAN           (score: 0.990)
  Inflation Reduction Act             -> LEGISLATION          (score: 0.985)

Text: The European Central Bank announced a new round of quantitative easing to support the eurozone economy.
  European Central Bank               -> FINANCIAL_ORG        (score: 0.968)
  eurozone                            -> GPE                  (score: 0.961)

Text: Christine Lagarde said the ECB remains committed to its 2% inflation target despite rising energy costs.
  Christine Lagarde                   -> POLITICIAN           (score: 0.865)
  ECB                                 -> FINANCIAL_ORG        (score:

## 11. Login HuggingFace

Inainte de upload, trebuie sa te autentifici. Daca esti in Colab, ruleaza celula urmatoare
si introdu token-ul tau de la https://huggingface.co/settings/tokens (cu permisiune **write**).

In [ ]:
from huggingface_hub import notebook_login, HfApi

notebook_login()
# Verificare ca login-ul a reusit
api = HfApi()
user_info = api.whoami()
print(f"\nAutentificat ca: {user_info['name']}")


Autentificat ca: Tudorx95


## 12. Upload dataset pe HuggingFace

Repo: `Tudorx95/NER_Political_Economic` (pe care l-ai creat deja).

In [ ]:
from huggingface_hub import HfApi, create_repo

api = HfApi()

# Cream repo daca nu exista (idempotent — exist_ok=True nu suprascrie)
try:
    create_repo(HF_DATASET_REPO, repo_type="dataset", exist_ok=True)
    print(f"Repo dataset OK: https://huggingface.co/datasets/{HF_DATASET_REPO}")
except Exception as e:
    print(f"Eroare la create_repo (probabil exista deja): {e}")

Repo dataset OK: https://huggingface.co/datasets/Tudorx95/NER_Political_Economic


In [ ]:
# ─── Upload fisierele dataset ────────────────────────────────────────────────
# Incarcam splits + statistici + sursele intermediare (utile pentru reproducibilitate)

files_to_upload = [
    (SPLITS_DIR / "train.jsonl",                     "splits/train.jsonl"),
    (SPLITS_DIR / "dev.jsonl",                       "splits/dev.jsonl"),
    (SPLITS_DIR / "test.jsonl",                      "splits/test.jsonl"),
    (BASE_DIR  / "dataset_stats.json",               "dataset_stats.json"),
    (BASE_DIR  / "annotated/weak_supervision.jsonl", "annotated/weak_supervision.jsonl"),
    (BASE_DIR  / "annotated/synthetic.jsonl",        "annotated/synthetic.jsonl"),
    (BASE_DIR  / "external/conll_pool.jsonl",        "external/conll_pool.jsonl"),
    (BASE_DIR  / "external/conll_gold.jsonl",        "external/conll_gold.jsonl"),
    (BASE_DIR  / "external/wnut_pool.jsonl",         "external/wnut_pool.jsonl"),
]

# Generam un README pentru dataset
dataset_readme = f"""---
license: mit
task_categories:
- token-classification
language:
- en
tags:
- ner
- politics
- economics
- weak-supervision
size_categories:
- 10K<n<100K
---

# NER Political & Economic Dataset

Custom NER dataset for politico-economic entity recognition, built using:
- **Pre-annotated sources:** CoNLL-2003, WNUT-2017 (re-mapped via gazetteers)
- **Weak supervision:** Snorkel labeling functions over CC-News, Wikipedia, SEC EDGAR
- **Synthetic templates** for rare classes

## Schema (11 labels)

`POLITICIAN`, `POLITICAL_PARTY`, `POLITICAL_ORG`, `FINANCIAL_ORG`,
`ECONOMIC_INDICATOR`, `POLICY`, `LEGISLATION`, `MARKET_EVENT`,
`CURRENCY`, `TRADE_AGREEMENT`, `GPE`

## Splits

- `splits/train.jsonl` — training set
- `splits/dev.jsonl`   — validation set
- `splits/test.jsonl`  — test set (includes CoNLL-2003 test as gold standard)

## Format

Each line is a JSON object:
```json
{{
    "text": "The Federal Reserve raised rates.",
    "entities": [
        {{"text": "Federal Reserve", "label": "FINANCIAL_ORG", "start": 4, "end": 19}}
    ],
    "source": "weak_supervision_cc_news"
}}
```

## Methodology

Inspired by Lison et al. (2020), "Named Entity Recognition without Labelled Data:
A Weak Supervision Approach" (ACL 2020).
"""

readme_path = BASE_DIR / "README_dataset.md"
with open(readme_path, "w") as f:
    f.write(dataset_readme)
files_to_upload.append((readme_path, "README.md"))

# Upload fiecare fisier
for local_path, repo_path in files_to_upload:
    if not local_path.exists():
        print(f"  SKIP (nu exista): {local_path}")
        continue
    print(f"  Upload: {repo_path} ({local_path.stat().st_size / 1024:.1f} KB)")
    api.upload_file(
        path_or_fileobj=str(local_path),
        path_in_repo=repo_path,
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
    )

print(f"\nDataset uploaded la: https://huggingface.co/datasets/{HF_DATASET_REPO}")

  Upload: splits/train.jsonl (1786.4 KB)


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


  Upload: splits/dev.jsonl (357.2 KB)


No files have been modified since last commit. Skipping to prevent empty commit.


  Upload: splits/test.jsonl (611.8 KB)
  Upload: dataset_stats.json (2.7 KB)


...

  Upload: external/conll_gold.jsonl (339.7 KB)


No files have been modified since last commit. Skipping to prevent empty commit.


  Upload: external/wnut_pool.jsonl (291.7 KB)


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


  Upload: README.md (1.2 KB)

Dataset uploaded la: https://huggingface.co/datasets/Tudorx95/NER_Political_Economic


## 13. Upload model GLiNER pe HuggingFace

Repo: `Tudorx95/NER_Economic_Political` (pe care l-ai creat deja).

In [ ]:
# Cream repo model (idempotent)
try:
    create_repo(HF_MODEL_REPO, repo_type="model", exist_ok=True)
    print(f"Repo model OK: https://huggingface.co/{HF_MODEL_REPO}")
except Exception as e:
    print(f"Eroare la create_repo (probabil exista deja): {e}")

Repo model OK: https://huggingface.co/Tudorx95/NER_Economic_Political


In [ ]:
# ─── Generam README pentru model ─────────────────────────────────────────────
# Citim metricile salvate ca sa le includem in README

with open(MODEL_OUT_DIR / "metrics.json") as f:
    m = json.load(f)

metrics_table_lines = [
    "| Label | Precision | Recall | F1 |",
    "|-------|-----------|--------|----|"
]
for label in NER_LABELS:
    if label in m["per_label_metrics"]:
        ent = m["per_label_metrics"][label]["ent_type"]
        metrics_table_lines.append(
            f"| {label} | {ent['precision']:.3f} | {ent['recall']:.3f} | {ent['f1']:.3f} |"
        )
metrics_table = "\n".join(metrics_table_lines)

g = m["global_metrics"]["ent_type"]

model_readme = f"""---
license: mit
language:
- en
library_name: gliner
pipeline_tag: token-classification
tags:
- ner
- gliner
- politics
- economics
base_model: {PRETRAINED_MODEL}
datasets:
- {HF_DATASET_REPO}
---

# GLiNER Fine-tuned for Political & Economic NER

Fine-tuned version of [`{PRETRAINED_MODEL}`]({PRETRAINED_MODEL}) on a custom
politico-economic NER dataset. Trained to recognize 11 entity types.

## Entity types

`POLITICIAN`, `POLITICAL_PARTY`, `POLITICAL_ORG`, `FINANCIAL_ORG`,
`ECONOMIC_INDICATOR`, `POLICY`, `LEGISLATION`, `MARKET_EVENT`,
`CURRENCY`, `TRADE_AGREEMENT`, `GPE`

## Performance

Test set: {m['num_test_examples']} examples.
Evaluation mode: `ent_type` (label match, ignoring exact boundaries).

**Global (micro-averaged):**
- Precision: **{g['precision']:.4f}**
- Recall:    **{g['recall']:.4f}**
- F1:        **{g['f1']:.4f}**

**Per label:**

{metrics_table}

## Usage

```python
from gliner import GLiNER

model = GLiNER.from_pretrained("{HF_MODEL_REPO}")
labels = ["POLITICIAN", "POLITICAL_PARTY", "POLITICAL_ORG", "FINANCIAL_ORG",
          "ECONOMIC_INDICATOR", "POLICY", "LEGISLATION", "MARKET_EVENT",
          "CURRENCY", "TRADE_AGREEMENT", "GPE"]

text = "The Federal Reserve raised rates after President Biden signed the new bill."
entities = model.predict_entities(text, labels, threshold=0.5)
for e in entities:
    print(e["text"], "->", e["label"])
```

## Training details

- Base model: `{PRETRAINED_MODEL}`
- Training examples: {m['num_train_examples']}
- Validation examples: {m['num_dev_examples']}
- Epochs: {m['training_config']['num_epochs']}
- Batch size: {m['training_config']['batch_size']}
- Learning rate: {m['training_config']['learning_rate']}
"""

model_readme_path = MODEL_OUT_DIR / "README.md"
with open(model_readme_path, "w") as f:
    f.write(model_readme)
print("README model generat.")

README model generat.


In [ ]:
# ─── Upload model files ──────────────────────────────────────────────────────
# GLiNER salveaza modelul ca un folder cu config + greutati. Le incarcam pe toate.

print(f"Continut {final_model_path}:")
for p in sorted(final_model_path.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(final_model_path)}  ({p.stat().st_size / 1e6:.1f} MB)")

# Upload intregul folder model
print(f"\nIncarcam folderul model in repo {HF_MODEL_REPO}...")
api.upload_folder(
    folder_path=str(final_model_path),
    repo_id=HF_MODEL_REPO,
    repo_type="model",
    commit_message="Upload fine-tuned GLiNER model",
)

# Upload README si metrics.json separat
api.upload_file(
    path_or_fileobj=str(model_readme_path),
    path_in_repo="README.md",
    repo_id=HF_MODEL_REPO,
    repo_type="model",
)
api.upload_file(
    path_or_fileobj=str(MODEL_OUT_DIR / "metrics.json"),
    path_in_repo="metrics.json",
    repo_id=HF_MODEL_REPO,
    repo_type="model",
)

print(f"\nModel uploaded la: https://huggingface.co/{HF_MODEL_REPO}")

Continut /content/drive/MyDrive/NER_Project/gliner_finetuned/final:
  gliner_config.json  (0.0 MB)
  pytorch_model.bin  (610.6 MB)
  tokenizer.json  (8.3 MB)
  tokenizer_config.json  (0.0 MB)

Incarcam folderul model in repo Tudorx95/NER_Economic_Political...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...d/final/pytorch_model.bin:   0%|          | 11.5kB /  611MB            


Model uploaded la: https://huggingface.co/Tudorx95/NER_Economic_Political


## 14. Verificare finala — incarcare model direct de pe HuggingFace

Ca sa fim siguri ca upload-ul a mers, descarcam modelul de pe HF si rulam o predictie.

In [ ]:
# Descarcam modelul de pe HuggingFace si testam
from gliner import GLiNER

print(f"Se descarca {HF_MODEL_REPO} de pe HuggingFace...")
hf_model = GLiNER.from_pretrained(HF_MODEL_REPO)
hf_model.to(DEVICE)

text = "The IMF warned that rising US interest rates could trigger a global recession."
entities = hf_model.predict_entities(text, NER_LABELS, threshold=EVAL_THRESHOLD)

print(f"\nText: {text}\n")
print("Entitati detectate:")
for e in entities:
    print(f"  {e['text']:<30} -> {e['label']:<20} (score: {e['score']:.3f})")

print("\n=== TOTUL OK ===")
print(f"Dataset:  https://huggingface.co/datasets/{HF_DATASET_REPO}")
print(f"Model:    https://huggingface.co/{HF_MODEL_REPO}")

Se descarca Tudorx95/NER_Economic_Political de pe HuggingFace...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]


Text: The IMF warned that rising US interest rates could trigger a global recession.

Entitati detectate:
  IMF                            -> FINANCIAL_ORG        (score: 0.998)
  US                             -> GPE                  (score: 1.000)

=== TOTUL OK ===
Dataset:  https://huggingface.co/datasets/Tudorx95/NER_Political_Economic
Model:    https://huggingface.co/Tudorx95/NER_Economic_Political
